In [2]:
from common.mistral import call_mistral, MistralCallConfig
from common.logger import JUPYTER_LOGGER as logger
from common.paths import get_sber_gitignore_data_dpath

import json
from typing import *
import pandas as pd

In [3]:
features_dpath = get_sber_gitignore_data_dpath() / "output"
features_fpaths: list = sorted(features_dpath.glob("*.csv"))
all_features = []
for fpath in features_fpaths:
    df = pd.read_csv(fpath)
    all_features.append(df)
features_df = pd.concat(all_features, ignore_index=True)
logger.info(f"Размер итогового датафрейма признаков: {features_df.shape}")

2026-03-30 23:33:31,260 - jupyter-notebooks - INFO - [JUPYTER 📓] Размер итогового датафрейма признаков: (560, 1620)


In [4]:
judge_prompt_template = """
Ты — судья, оценивающий точность ответа модели.
Галлюцинация — это когда модель генерирует информацию, не соответствующую фактам и недостоверную.

Вопрос: {query}
Правильный ответ (достоверный эталон): {ground_truth}
Ответ модели: {model_answer}

Определи, является ли ответ модели галлюцинацией.
Выбери строго один вариант и ничего больше: "галлюцинация" или "не галлюцинация".
ТОЛЬКО эти два слова, без кавычек и без дополнительных пояснений.
"""
prompts = [judge_prompt_template.format(query=row["query"], ground_truth=row["ground_truth"], model_answer=row["model_answer"]) for _, row in features_df.iterrows()]
features_df["judge_prompt"] = prompts

C:\Users\User\AppData\Local\Temp\ipykernel_14972\3639616577.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features_df["judge_prompt"] = prompts


In [5]:
from tqdm.auto import tqdm

mistral_cfg = MistralCallConfig(
    models_list=["mistral-small-2603"],
)

judge_system_prompt = (
    "Ты строгий факт-чекер и судья качества ответов. "
    "Проверяй соответствие ответа модели эталонному ответу. "
    "Если доступен внешний инструмент поиска, используй его перед финальным вердиктом. "
    "Отвечай только в требуемом JSON-формате."
)

# Для детерминированной и воспроизводимой оценки.
judge_generation_kwargs = {
    "temperature": 0.1,
    "top_p": 0.9,
    "presence_penalty": 0.0,
    "frequency_penalty": 0.0,
    "reasoning_effort": "high",
    "response_format": {"type": "json_object"},
}

# При наличии интеграции можно передать tools (например, web-search/knowledge-base).
# Если tools пустой, вызов остаётся совместимым с текущим пайплайном.
judge_tools: list[dict[str, Any]] = []
judge_tool_choice: str = "required" if judge_tools else "auto"

output_fpath = get_sber_gitignore_data_dpath() / "judge_scores.csv"
scores = []
queries = []
gt = []
answers = []

def save_scores():
    scores_df = {
        "query": queries,
        "ground_truth": gt,
        "model_answer": answers,
        "judge_score": scores,
    }
    pd.DataFrame(scores_df).to_csv(output_fpath, index=False)


def _normalize_judge_score(raw_response: str) -> str:
    text = raw_response.strip().lower()

    # Предпочитаем структурированный ответ из response_format=json_object.
    try:
        parsed: Any = json.loads(text)
        if isinstance(parsed, dict):
            score_value: Any = parsed.get("judge_score")
            if isinstance(score_value, str):
                normalized = score_value.strip().lower()
                if normalized in {"галлюцинация", "не галлюцинация"}:
                    return normalized
    except json.JSONDecodeError:
        pass

    if "не галлюцинация" in text:
        return "не галлюцинация"
    if "галлюцинация" in text:
        return "галлюцинация"
    return "неизвестно"


def _build_call_kwargs(prompt: str) -> dict[str, Any]:
    messages: list[dict[str, str]] = [
        {"role": "system", "content": judge_system_prompt},
        {"role": "user", "content": prompt},
    ]
    call_kwargs: dict[str, Any] = {
        "messages": messages,
        **judge_generation_kwargs,
    }
    if judge_tools:
        call_kwargs["tools"] = judge_tools
        call_kwargs["tool_choice"] = judge_tool_choice
    return call_kwargs

def make_call(raw):
    prompt = raw.judge_prompt
    call_kwargs = _build_call_kwargs(prompt)
    response = call_mistral(mistral_cfg, **call_kwargs)
    score = _normalize_judge_score(response)
    scores.append(score)
    queries.append(raw.query)
    gt.append(raw.ground_truth)
    answers.append(raw.model_answer)
    save_scores()

# test call
test_raw = features_df.iloc[0]
make_call(test_raw)
test_score = scores[-1]
logger.info(f"Тестовый вызов завершился. Промпт:\n{test_raw.judge_prompt}\nОтвет модели: {test_raw.model_answer}\nОценка судьи: {test_score}")

2026-03-30 23:33:34,799 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: начало вызова API
2026-03-30 23:33:34,799 - mistral-call - DEBUG - [MISTRAL 🇫🇷] call_mistral: параметры - ['messages', 'temperature', 'top_p', 'presence_penalty', 'frequency_penalty', 'reasoning_effort', 'response_format']
2026-03-30 23:33:34,799 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: попытка 1/1 с моделью mistral-small-2603, ключ 1/13
2026-03-30 23:33:35,180 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: вызов функции с timeout=240
2026-03-30 23:33:35,180 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: попытка 1
2026-03-30 23:33:35,180 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: попытка вызова модели mistral-small-2603
2026-03-30 23:33:46,520 - mistral-call - ERROR - [MISTRAL 🇫🇷] safe_call: ошибка на попытке 1: API error occurred: ...
Traceback (most recent call last):
  File "C:\Users\User\Desktop\dirs\Dev\hack-mfti\common\mistral.py", line 164, in safe_call
    result = future.result(timeou

In [ ]:
pbar = tqdm(total=len(features_df), desc="Оценка ответов судьей")
for _, row in features_df.iterrows():
    make_call(row)
    pbar.update(1)
pbar.close()
logger.info(f"Оценка всех ответов завершена. Результаты сохранены в {output_fpath}")

Оценка ответов судьей:   0%|          | 0/560 [00:00<?, ?it/s]

2026-03-30 23:34:47,091 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: начало вызова API
2026-03-30 23:34:47,092 - mistral-call - DEBUG - [MISTRAL 🇫🇷] call_mistral: параметры - ['messages', 'temperature', 'top_p', 'presence_penalty', 'frequency_penalty', 'reasoning_effort', 'response_format']
2026-03-30 23:34:47,095 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: попытка 1/1 с моделью mistral-small-2603, ключ 2/13
2026-03-30 23:34:47,115 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: вызов функции с timeout=240
2026-03-30 23:34:47,117 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: попытка 1
2026-03-30 23:34:47,119 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: попытка вызова модели mistral-small-2603
2026-03-30 23:34:47,755 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: получен ответ длиной 27 символов
2026-03-30 23:34:47,757 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: успешное выполнение на попытке 1
2026-03-30 23:34:47,759 - mistral-call - INFO - [MISTRAL 🇫🇷] call_m